<a href="https://colab.research.google.com/github/hslein/PDI-2025-2/blob/main/ProyectoFinalPDI1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

In [2]:
!pip install ultralytics

In [3]:
import os
import re
import glob
import time
import random
import numpy as np
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from roboflow import Roboflow
import matplotlib.pyplot as plt
from huggingface_hub import notebook_login

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

notebook_login()

In [4]:
HF_USER = "hslein"
SPACE_NAME = "PDI-1"
REPO_URL = f"https://huggingface.co/spaces/hslein/PDI-1"

In [6]:
!git clone {REPO_URL}

fatal: destination path 'PDI-1' already exists and is not an empty directory.


In [7]:
%cd PDI-1
!git lfs install
!git lfs track "*.pt"
!git add .gitattributes
!git commit -m "enable git lfs for YOLO model"


/content/PDI-1
Updated git hooks.
Git LFS initialized.
"*.pt" already supported
On branch main
Your branch and 'origin/main' have diverged,
and have 1 and 2 different commits each, respectively.
  (use "git pull" to merge the remote branch into yours)

nothing to commit, working tree clean


In [12]:
!cp best.pt PDI-1/

In [8]:
%%writefile {SPACE_NAME}/app.py
from fastapi import FastAPI
from pydantic import BaseModel
import base64
import numpy as np
from PIL import Image
import io
from ultralytics import YOLO

app = FastAPI(title="YOLO TACO Deployment")
MODEL_PATH = "best.pt"
model = YOLO(MODEL_PATH)

class ImagePayload(BaseModel):
  image_base64: str

@app.get("/")
def home():
  return {
    "status": "ok",
    "message": "YOLOv8 API is running! Use POST /predict",
    "model": str(model),
  }

def decode_image(base64_str):
  """Decodifica base64 → PIL Image"""
  img_bytes = base64.b64decode(base64_str)
  img = Image.open(io.BytesIO(img_bytes))

  if img.mode != "RGB":
    img = img.convert("RGB")

  return img

def encode_image(img_array):
  """Codifica imagen numpy → base64"""
  pil_img = Image.fromarray(img_array)
  buffer = io.BytesIO()
  pil_img.save(buffer, format="PNG")
  return base64.b64encode(buffer.getvalue()).decode()

@app.post("/predict")
def predict(payload: ImagePayload):
  """
  Recibe imagen en base64.
  Ejecuta YOLO.
  Devuelve:
    - clases detectadas
    - bounding boxes
    - imagen anotada (base64)
  """
  try:
    img = decode_image(payload.image_base64)
    results = model.predict(img)
    r = results[0]

    # Imagen anotada
    annotated = r.plot()  # numpy array BGR
    annotated = annotated[:, :, ::-1]  # convertir BGR → RGB

    encoded_annotated = encode_image(annotated)

    # Crear lista de detecciones
    detections = []
    for box in r.boxes:
      cls = int(box.cls[0])
      confidence = float(box.conf[0])
      xyxy = box.xyxy[0].tolist()

      detections.append({
        "class_id": cls,
        "class_name": model.names[cls],
        "confidence": confidence,
        "bbox_xyxy": xyxy
      })

    # Respuesta
    return {
      "status": "success",
      "detections": detections,
      "image_annotated_base64": encoded_annotated,
      "num_objects": len(detections)
    }

  except Exception as e:
    return {
      "status": "error",
      "message": str(e)
    }


@app.get("/health")
def health_check():
  return {
    "status": "healthy",
    "model_loaded": True
  }

Overwriting PDI-1/app.py


In [13]:
%%writefile {SPACE_NAME}/runtime.txt
python-3.10

Overwriting PDI-1/runtime.txt


In [14]:
!git config --global user.email "hslein@unal.edu.co"
!git config --global user.name "hslein"

In [15]:
%cd content

[Errno 2] No such file or directory: 'content'
/content


In [26]:
%%writefile /content/{SPACE_NAME}/Dockerfile
FROM python:3.10

# Fix OpenCV dependencies
RUN apt-get update && apt-get install -y \
    libgl1 \
    libgl1-mesa-dev \
    libglib2.0-0 \
    libsm6 \
    libxext6 \
    libxrender1

# Crear usuario no-root (requerido por HuggingFace)
RUN useradd -m user
USER user
ENV PATH="/home/user/.local/bin:${PATH}"

WORKDIR /app

# Copiar requirements
COPY --chown=user ./requirements.txt ./requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copiar código fuente
COPY --chown=user ./ ./

# Comando para ejecutar FastAPI
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"]

Overwriting /content/PDI-1/Dockerfile


In [27]:
%cd {SPACE_NAME}
!git add .
!git commit -m "Docker added1."
!git push -f

[Errno 2] No such file or directory: 'PDI-1'
/content/PDI-1
[main c94a268] Docker added1.
 1 file changed, 1 insertion(+), 1 deletion(-)
Enumerating objects: 35, done.
Counting objects: 100% (35/35), done.
Delta compression using up to 2 threads
Compressing objects: 100% (33/33), done.
Writing objects: 100% (35/35), 7.55 KiB | 1.26 MiB/s, done.
Total 35 (delta 12), reused 0 (delta 0), pack-reused 0
To https://huggingface.co/spaces/hslein/PDI-1
 + cf8f0dd...c94a268 main -> main (forced update)
